In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
gemeni_api_key = os.environ["GEMINI_API_KEY"]

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash"
)

In [6]:
# prompt templete
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
prompt = PromptTemplate.from_template(
    "Explain {topic} in simple language with an example."
)

chain = prompt | llm

response = chain.invoke({
    "topic": "Retrieval Augmented Generation"
})

In [7]:
response

AIMessage(content=[{'type': 'text', 'text': 'Imagine you are taking a difficult test.\n\n* **Regular AI** is like taking a **closed-book test**. It has to rely entirely on what it memorized months ago during training. If it doesn\'t know the answer, or if the information is brand-new, it might guess or make something up (this is called "hallucinating").\n* **RAG (Retrieval-Augmented Generation)** is like taking an **open-book test**. Instead of relying only on memory, the AI is allowed to look through a stack of reference books, find the exact page with the right information, and use it to write the answer.\n\n---\n\n### How It Works (The 3 Steps)\n\nBreak down the name to understand the process:\n\n1. **Retrieval (Search):** When you ask a question, the system first searches a specific database, document library, or the internet for relevant facts.\n2. **Augmentation (Add):** It takes the facts it found and "attaches" them to your original question behind the scenes.\n3. **Generation 

In [8]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "¡hola!"},
    {"input": "bye!", "output": "¡adiós!"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Spanish translator."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)


chain = final_prompt | llm

res = chain.invoke({"input": "How are you?"})
print(res.content)

[{'type': 'text', 'text': '¿Cómo estás?', 'extras': {'signature': 'ErcECrQEAWkUfRMbeIS5gKuZOHrrVVnNl6hzYNG5EwlC/B+QOynw737qYQFllMvA93SW4Vg+86fX2r4cNLqpfrLjGbbEfgUW5+dyOVA7sQdENrbrpJZBkmvEuineFHiEkBqyziZbXEHQU87/jkSkPRYKQ04jY3BNGFzngv9JNZM2UX6KgRt2V5FjQ2zDp63kM/KKJjn2q5O1gNf8vh37MHW4qT8VxFDg2qE9dbsgkWjeb2H9QnPYZYw7Zy4ACrT9zKdgaUVFQoGMBj2v0i9RCYs+7m2ze1b/rXEocj2LHcM3xX7jFRr3aMT/SvU26s3dIxAbgpCgcViqLUAzR77L/xWZJbGx0Q+cHgBnUokEJiKcfFfDR9yF70DCIth376H3KB6u/87bNbBVvQkw++5BuFtqT0FO349K95yYC0ma7A+Ti//fCQDexdvOe7pfHflOT08e2tgsXiYSbiQ4n5sAmptdzLZG2mhaDyZKU2T5fZs+51eKx/1zg8HCkqlMeLel9S+trFnNJr712kFz3BcgwCWz192MJkqJI0p/vyviMImQnhFN57wJbnEVQvYSN/f8jRN6M2Bc2WkEokhvbNPEmZrMw53/7uKE67uiZuFDfdd21aJ++cHteS+DJy/jg2I8Uk35F7ocfk6sretPUnAY0PhbuTOYBuCLiVfczcPxPPjcTRo5uY3VBmNvGxXR0/atEnYHePCR9XNnwln8wjykQF+w8Xxv+tsdhTWdYgrNGNxrmkMuC5so6pq6'}}]


In [ ]:
# output perser
from langchain_core.output_parsers import JsonOutputParser

In [13]:
json_prompt = PromptTemplate.from_template(
    "Return a JSON object with an `answer` key that answers the following question: {question}"
)
json_parser = JsonOutputParser()

In [14]:
json_chain = json_prompt | llm | json_parser

In [15]:
json_parser.get_format_instructions()

'Return a JSON object.'

In [16]:
res  = json_chain.invoke({"question": "What is the biggest country?"})

In [17]:
res

{'answer': 'Russia'}

In [37]:
from pydantic import BaseModel, Field
class Joke(BaseModel):
    setup: str = Field(description="question to set up a story")
    punchline: str = Field(description="answer to resolve the story")

In [23]:
# Define the parser referring the Pydantic Object
parser = JsonOutputParser(pydantic_object=Joke)
parser

JsonOutputParser(pydantic_object=<class '__main__.Joke'>)

In [38]:
# Add the parser format instructions in the prompt definition.
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

In [39]:
chain = prompt | llm | parser

In [40]:
chain.invoke({"query": "Tell me a story."})

{'setup': 'Did you hear the story about the broken pencil?',
 'punchline': 'Never mind, it has no point.'}